In [ ]:
from IPython.display import display, HTML
display(HTML("""
<style>
@import url('https://fonts.googleapis.com/css2?family=DM+Sans:wght@300;400;500;600;700&family=DM+Mono:wght@400;500&display=swap');
body, p, li, span, div { font-family: 'DM Sans', sans-serif !important; color: #0d0d0d; }
.nb-header { border-left: 4px solid #006d5b; padding: 1.2rem 1.5rem; background: #f7f7f5; border-radius: 0 8px 8px 0; margin-bottom: 1.5rem; }
.nb-header h1 { font-size: 1.6rem; font-weight: 700; margin: 0 0 0.3rem 0; letter-spacing: -0.5px; color: #0d0d0d; }
.nb-header p { margin: 0; color: #555; font-size: 0.95rem; }
.nb-step { background: white; border: 1.5px solid #e8e8e8; border-radius: 8px; padding: 1.2rem 1.5rem; margin-bottom: 1rem; }
.nb-step h3 { font-size: 1rem; font-weight: 700; text-transform: uppercase; letter-spacing: 0.5px; color: #006d5b; margin: 0 0 0.4rem 0; }
.nb-step p { margin: 0; color: #333; font-size: 0.9rem; line-height: 1.5; }
.nb-instructions { background: #f7f7f5; border-radius: 8px; padding: 1rem 1.5rem; margin-bottom: 1.5rem; }
.nb-instructions ol { margin: 0.5rem 0 0 0; padding-left: 1.2rem; }
.nb-instructions li { margin-bottom: 0.3rem; font-size: 0.9rem; color: #333; }
.nb-instructions strong { color: #0d0d0d; }
</style>
<div class='nb-header'>
  <h1>🔬 Protein Sequence Mapping</h1>
  <p>Universal Metabolic Community Pipeline &mdash; Step 2</p>
</div>
<div class='nb-instructions'>
  <strong>How to run this notebook:</strong>
  <ol>
    <li>Go to <strong>Runtime &rarr; Run all</strong> in the top menu</li>
    <li>Upload your files when prompted in Step 2</li>
    <li>Wait for BLAST to finish (a few minutes)</li>
    <li>Download the CSV from the <strong>Files panel on the left</strong> (&#128193;) &rarr; right-click &rarr; Download</li>
    <li>Upload the CSV to Step 2 of the pipeline</li>
  </ol>
</div>
"""))

In [ ]:
#@title Step 1 — Install BLAST { display-mode: "form" }
from IPython.display import display, HTML
display(HTML("<div class='nb-step'><h3>Step 1 &mdash; Install BLAST</h3><p>Installing required tools. This runs automatically and takes about 1 minute.</p></div>"))
import shutil, subprocess
if not shutil.which('blastp'):
    print('Installing BLAST...')
    subprocess.run(['apt-get', 'install', '-y', '-q', 'ncbi-blast+'], check=True)
    print('✅ BLAST installed.')
else:
    print('✅ BLAST already installed.')
subprocess.run(['pip', 'install', '-q', 'pandas'], capture_output=True)

In [ ]:
#@title Step 2 — Upload FASTA files { display-mode: "form" }
from IPython.display import display, HTML
from google.colab import files
import os

display(HTML("<div class='nb-step'><h3>Step 2 &mdash; Upload your FASTA files</h3><p>Upload your <strong>query FASTA</strong> (model protein sequences) then your <strong>database FASTA</strong> (NCBI protein sequences for the same species).</p></div>"))

print('📂  Upload QUERY FASTA — protein sequences from your metabolic model')
uploaded_query = files.upload()
query_filename_raw = list(uploaded_query.keys())[0]
query_filename = query_filename_raw.replace(' ', '_')
if query_filename != query_filename_raw:
    os.rename(query_filename_raw, query_filename)
print(f'✅ Query: {query_filename}')

print('\n📂  Upload DATABASE FASTA — protein sequences downloaded from NCBI')
uploaded_db = files.upload()
db_filename_raw = list(uploaded_db.keys())[0]
db_filename = db_filename_raw.replace(' ', '_')
if db_filename != db_filename_raw:
    os.rename(db_filename_raw, db_filename)
print(f'✅ Database: {db_filename}')

species_name = query_filename.split('_')[0]
display(HTML(f"<div style='margin-top:0.8rem;padding:0.7rem 1rem;background:#006d5b;color:#ffffff;border-radius:6px;font-family:DM Sans,sans-serif;font-size:0.9rem;font-weight:600;'>🔬 Species detected: {species_name}</div>"))

In [ ]:
#@title Step 3 — Run BLAST { display-mode: "form" }
from IPython.display import display, HTML
import subprocess, pandas as pd, os

display(HTML("<div class='nb-step'><h3>Step 3 &mdash; Run BLAST &amp; generate mapping CSV</h3><p>Running BLASTp and filtering results. This may take several minutes.</p></div>"))

db_name    = f'{species_name}_db'
blast_out  = f'{species_name}_blast.txt'
output_csv = f'{species_name}_protein_id_mapping.csv'

print('⚙️  Building BLAST database...')
subprocess.run(['makeblastdb', '-in', db_filename, '-dbtype', 'prot', '-out', db_name],
               check=True, capture_output=True)

print('⚙️  Running BLASTp...')
subprocess.run([
    'blastp', '-query', query_filename, '-db', db_name, '-out', blast_out,
    '-outfmt', '6 qseqid sseqid pident length mismatch gapopen qstart qend sstart send evalue bitscore qcovs',
    '-num_threads', '2'
], check=True, capture_output=True)

cols = ['qseqid','sseqid','pident','length','mismatch','gapopen','qstart','qend','sstart','send','evalue','bitscore','qcovs']
blast_df = pd.read_csv(blast_out, sep='\t', header=None, names=cols)

def seq_lengths(fasta):
    lengths, cur_id, cur_len = {}, None, 0
    with open(fasta) as f:
        for line in f:
            if line.startswith('>'):
                if cur_id: lengths[cur_id] = cur_len
                cur_id, cur_len = line[1:].strip().split()[0], 0
            else:
                cur_len += len(line.strip())
    if cur_id: lengths[cur_id] = cur_len
    return lengths

blast_df['qlen'] = blast_df['qseqid'].map(seq_lengths(query_filename))
blast_df['slen'] = blast_df['sseqid'].map(seq_lengths(db_filename))
blast_df['qcov'] = (blast_df['qend'] - blast_df['qstart'] + 1) / blast_df['qlen']
blast_df['scov'] = blast_df['length'] / blast_df['slen']

filtered = blast_df[
    (blast_df['pident']   >= 95)   &
    (blast_df['gapopen']  == 0)    &
    (blast_df['mismatch'] <= 1)    &
    (blast_df['qcov']     >= 0.85) &
    (blast_df['scov']     >= 0.85)
]

best   = filtered.sort_values('evalue').groupby('qseqid').first().reset_index()
result = best[['qseqid','sseqid']].rename(columns={'qseqid':'identifier_query','sseqid':'identifier_db'})
result.to_csv(output_csv, index=False)

display(HTML(f"""
<div style='margin-top:1rem;padding:1rem 1.5rem;background:#f0fff8;border:1.5px solid #006d5b;border-radius:8px;font-family:DM Sans,sans-serif;'>
  <div style='font-size:1rem;font-weight:700;color:#006d5b;margin-bottom:0.4rem;'>✅ Done!</div>
  <div style='font-size:0.9rem;color:#333;'>{len(result)} matches found out of {blast_df['qseqid'].nunique()} sequences.</div>
  <div style='font-size:0.9rem;color:#333;margin-top:0.3rem;'>Output saved as <strong>{output_csv}</strong></div>
  <div style='font-size:0.9rem;color:#555;margin-top:0.6rem;'>&#128193; Find the file in the <strong>Files panel on the left</strong> &rarr; right-click &rarr; <strong>Download</strong></div>
</div>
"""))